In [27]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


In [28]:
import pandas as pd

train_path = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"
test_path = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet"

train = pd.read_parquet(train_path)
test = pd.read_parquet(test_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (2539608, 18)
Test shape: (1213, 12)


In [43]:
train.head(3)

,ingest_lib,normalized_smiles,inchikey,inchikey14,molecular_formula,ionization_mode,instrument_type,adduct,adduct_orig,precursor_mz,precursor_error_ppm,ms2_mzs,ms2_normalized_intensities,num_peaks,base_peak_intensity,collision_energy_ev,collision_energy_orig,collision_energy_orig_units
0,drug_plus,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA-UHFFFAOYSA-N,AAALVYBICLMAMA,C20H15N3O2,positive,None,[M+H]+,[M+H]+,330.1237,1.671398,"[92.0495, 93.0573, 94.0607, 116.107, 123.1168,...","[0.0171037726229424, 0.328091214993177, 0.0107...",52,NaN,None,None,unknown
1,drug_plus,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG-UHFFFAOYSA-N,AAKJLRGGTJKAMG,C22H23N3O4,positive,None,[M+K]+,[M+K]+,432.1320,1.302193,"[82.1451, 89.0599, 158.9638, 248.0836, 250.098...","[0.00144055962319783, 0.959302375462881, 0.5, ...",28,NaN,None,None,unknown
2,drug_plus,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR-UHFFFAOYSA-N,AANFHDFOMFRLLR,C23H26FNO4,positive,None,[M+Na]+,[M+Na]+,422.1738,1.316437,"[91.0539, 111.117, 131.0856, 150.035, 171.0803...","[0.00262159533923045, 0.0184752871374318, 0.00...",41,NaN,None,None,unknown


In [30]:
print(train.info(show_counts=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2539608 entries, 0 to 2539607
Data columns (total 18 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   ingest_lib                   2539608 non-null  object 
 1   normalized_smiles            2539608 non-null  object 
 2   inchikey                     2539608 non-null  object 
 3   inchikey14                   2539608 non-null  object 
 4   molecular_formula            2539608 non-null  object 
 5   ionization_mode              2539608 non-null  object 
 6   instrument_type              2506555 non-null  object 
 7   adduct                       2539608 non-null  object 
 8   adduct_orig                  2539608 non-null  object 
 9   precursor_mz                 2539608 non-null  float64
 10  precursor_error_ppm          2535322 non-null  float64
 11  ms2_mzs                      2539608 non-null  object 
 12  ms2_normalized_intensities   2539608 non-n

In [31]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1213 entries, 0 to 1212
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   molecule_id                  1213 non-null   object 
 1   spectrum_id                  1213 non-null   object 
 2   ms2_mzs                      1213 non-null   object 
 3   ms2_normalized_intensities   1213 non-null   object 
 4   base_peak_intensity          1213 non-null   float64
 5   adduct                       1213 non-null   object 
 6   ionization_mode              1213 non-null   object 
 7   instrument_type              1213 non-null   object 
 8   precursor_mz                 1213 non-null   float64
 9   collision_energy_orig        1213 non-null   object 
 10  collision_energy_ev          1213 non-null   object 
 11  collision_energy_orig_units  1213 non-null   object 
dtypes: float64(2), object(10)
memory usage: 113.8+ KB


In [32]:
print("m/z:", train["ms2_mzs"].iloc[0])
print("Intensity:", train["ms2_normalized_intensities"].iloc[0])

m/z: [ 92.0495  93.0573  94.0607 116.107  123.1168 124.1203 133.0761 142.0777
 142.1226 166.0655 167.0734 168.0808 169.0838 169.0886 170.092  176.0457
 181.076  184.0758 185.0709 188.0452 192.0683 193.076  194.06   201.0533
 209.071  212.0698 219.0554 221.072  225.0532 225.0658 234.0664 234.0664
 237.066  238.0736 240.0799 250.0615 251.0687 252.0765 254.0839 257.108
 259.1231 278.0925 284.1179 285.1262 286.1341 287.1178 290.1275 301.085
 302.1284 312.1133 314.0926 328.1069]
Intensity: [1.71037726e-02 3.28091215e-01 1.07218612e-02 2.60808387e-02
 6.89810237e-03 7.31897657e-04 2.96068467e-03 1.98119487e-03
 7.14763564e-04 1.93528596e-03 2.67409777e-03 4.00310632e-02
 1.53947495e-03 4.52608905e-02 4.92054957e-03 7.90908500e-04
 3.35415059e-03 1.11231631e-02 7.63319334e-04 8.34675223e-04
 1.38966398e-03 7.90655389e-03 5.82703635e-01 3.81112746e-03
 2.94816683e-02 8.85462555e-04 1.48710610e-01 9.45472234e-04
 2.72909823e-03 1.59775040e-03 9.60284483e-03 9.60284483e-03
 1.51136640e-01 2.1635

In [33]:
mz = train["ms2_mzs"].iloc[0]
intensity = train["ms2_normalized_intensities"].iloc[0]

print("m/z length:", len(mz))
print("Intensity length:", len(intensity))
print("Same length:", len(mz) == len(intensity))

m/z length: 52
Intensity length: 52
Same length: True


In [34]:
print("Maximum intensity:", intensity.max())
print("Minimum intensity:", intensity.min())

Maximum intensity: 1.0
Minimum intensity: 0.00059942967118094


In [35]:
import time
start_time = time.time()
max_intensities = train["ms2_normalized_intensities"].apply(max)

print("time taken ", time.time() - start_time)
print("Maximum of maximum intensities:", max_intensities.max())
print("Minimum of maximum intensities:", max_intensities.min())

time taken  30.701749563217163
Maximum of maximum intensities: 1.0
Minimum of maximum intensities: 1.0


In [42]:
mz_lengths = train["ms2_mzs"].apply(len)
intensity_lengths = train["ms2_normalized_intensities"].apply(len)

print("All lengths match:", (mz_lengths == intensity_lengths).all())

All lengths match: True


In [41]:
print(
    "All num_peaks match m/z length:",
    (train["num_peaks"] == train["ms2_mzs"].apply(len)).all()
)

All num_peaks match m/z length: True


In [45]:
pd.set_option("display.float_format", "{:,.2f}".format)
print(train["num_peaks"].describe())

count   2,539,608.00
mean          158.22
std           516.88
min             1.00
25%            14.00
50%            42.00
75%           164.00
max        73,318.00
Name: num_peaks, dtype: float64


In [46]:
mz = train["ms2_mzs"].iloc[0]

print("Minimum m/z:", mz.min())
print("Maximum m/z:", mz.max())

Minimum m/z: 92.0495
Maximum m/z: 328.1069
